# pf-variant-priority — walkthrough

Prioritising variants of unknown significance in *P. falciparum* drug-resistance panels.

This notebook runs **entry point B**: starting from a variant table rather than raw reads,
so it works without a sequencing run. It covers

1. loading a variant table and the panel proteins
2. scoring missense variants with ESM-2
3. calibrating the score against WHO-validated *kelch13* markers
4. producing a ranked watchlist, and reading it honestly

**Before you interpret anything:** check `benchmark.json`. If the score cannot recover
known markers, the watchlist is exploratory and nothing more.

## 0. Setup

```bash
conda env create -f ../environment.yml && conda activate pf-vp
```

Fetch the panel protein sequences from PlasmoDB (release ≥ 52). Headers must be the
PlasmoDB gene IDs used in `data/pfmdr15_panel.tsv`, e.g. `>PF3D7_1343700`.

In [ ]:
import sys, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "bin"))

panel = pd.read_csv(ROOT / "data/pfmdr15_panel.tsv", sep="\t", comment="#")
print(f"{len(panel)} panel genes")
panel.head(16)

## 1. Input variants

Replace the example file with real data. Options:

* **MalariaGEN Pf7/Pf8** via the `malariagen_data` package — open access, includes Indian sites
* **your own `bcftools csq` output** — run `bin/collect_variants.py` over the VCFs first

`data/example_variant_table.tsv` is synthetic smoke-test data. It is not real and must
not be interpreted.

In [ ]:
variants = pd.read_csv(ROOT / "data/example_variant_table.tsv", sep="\t", comment="#")
print(f"{len(variants)} variants across {variants.gene.nunique()} genes")
variants

### Loading real data from MalariaGEN

```python
import malariagen_data
pf = malariagen_data.Pf7()
meta = pf.sample_metadata()
india = meta[meta.country == "India"]
print(len(india), "Indian samples")
# then extract variant calls for the panel gene regions and aggregate
# into the same columns: gene, plasmodb_id, aa_ref, aa_pos, aa_alt,
# n_samples, n_total, allele_freq, n_sites, first_year, last_year
```

## 2. Score with ESM-2

Masked-marginal log-odds. Use the 35M checkpoint on a laptop, 650M on a GPU.

Long proteins are windowed (ESM takes ~1022 residues; pfubp1 is ~3800 aa) and
low-complexity regions are flagged rather than silently scored — asparagine repeats are
common in *P. falciparum* and language models handle them badly.

In [ ]:
cmd = [sys.executable, str(ROOT / "bin/score_variants.py"),
       "--variants", str(ROOT / "data/example_variant_table.tsv"),
       "--fasta",    str(ROOT / "data/panel_proteins.faa"),
       "--checkpoint", "facebook/esm2_t12_35M_UR50D",
       "--out",      str(ROOT / "results/scored.tsv")]
print(" ".join(cmd))
# subprocess.run(cmd, check=True)   # uncomment once panel_proteins.faa is in place

## 3. Calibrate against known markers

The 13 WHO-validated *kelch13* markers are the positive set. Everything else observed in
*kelch13* that is neither validated nor a WHO candidate is the negative set.

Read the caveat in the output: negatives are *not yet known to matter*, not *known not to
matter*, so AUC is biased downwards. Thirteen positives in one structural domain is a thin
benchmark, hence the bootstrap interval.

In [ ]:
!python {ROOT}/bin/benchmark.py --scored {ROOT}/results/scored.tsv \
    --validated {ROOT}/data/who_validated_k13.tsv --out {ROOT}/results/benchmark/

import json
report = json.loads((ROOT / "results/benchmark/benchmark.json").read_text())
print(json.dumps(report, indent=2))

### How to read the AUC

| AUC | What it means | What to do |
|---|---|---|
| ~0.5, CI spans 0.5 | no signal | do not use the watchlist for triage; say so |
| 0.6–0.7 | weak | use only alongside other evidence |
| >0.7, CI excludes 0.5 | usable | cautious triage, still needs experimental follow-up |

If `status` is `INSUFFICIENT_DATA`, too few markers were scored to say anything.

## 4. Ranked watchlist

Constraint, allele frequency, geographic spread and temporal trend, combined with the
hand-set weights in `conf/priority.yml`. The weights are not learned — there is no
labelled set of Indian VUS to learn them from — so `prioritise.py` reports a rank
stability interval from perturbing them.

In [ ]:
!python {ROOT}/bin/prioritise.py --scored {ROOT}/results/scored.tsv \
    --config {ROOT}/conf/priority.yml --out {ROOT}/results/watchlist.tsv

wl = pd.read_csv(ROOT / "results/watchlist.tsv", sep="\t", comment="#")
cols = ["rank", "gene", "aa_ref", "aa_pos", "aa_alt", "allele_freq",
        "esm_score", "priority_score", "rank_p05", "rank_p95", "rank_stable"]
wl[cols].head(15)

### Sensitivity to the weights

Only variants with `rank_stable == True` keep roughly the same position when the weights
are perturbed. Treat the rest as unordered.

In [ ]:
stable = wl[wl.rank_stable]
print(f"{len(stable)}/{len(wl)} variants have stable ranks")
stable[["rank", "gene", "aa_ref", "aa_pos", "aa_alt", "priority_score"]].head(10)

## 5. What this output is, and is not

**Is:** a reproducible triage ordering, telling you which unvalidated variants deserve
limited follow-up effort first.

**Is not:** a prediction of drug resistance. ESM scores evolutionary plausibility, not drug
phenotype. A highly constrained position may be essential for parasite viability and
therefore *less* likely to carry a viable resistance allele.

**Next steps:** link to *in vitro* IC50 or clinical outcome data; add AlphaFold3 structural
context; fine-tune a language model on apicomplexan proteomes; extend the framework to
*Anopheles stephensi* insecticide-resistance loci.